In [25]:
import numpy as np
import heapq
import pandas as pd
from scipy.stats import skewnorm
from sklearn.cluster import KMeans
import os


In [ ]:

# =====================================================
# PARAMETERS
# =====================================================

SIM_TIME             = 45 * 24
DRIVER_RATE          = 4.74
RIDER_RATE           = 34.6
PATIENCE_RATE        = 5
SPEED                = 20
AREA_SIZE            = 20
BASE_FARE            = 3
FARE_PER_MILE        = 2
PETROL_COST_PER_MILE = 0.20
CVAR_ALPHA           = 0.05

LAM_VALUES  = np.linspace(1, 4.9, 40).tolist()   # lambda values to sweep
K_VALUES    = range(3, 5)                  # K=1 to K=10

OUTPUT_DIR  = "run_out_of_name"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [27]:

# =====================================================
# STEP 1 — BUILD KMEANS HUB CONFIGS FROM REAL DATA
# =====================================================

def parse_coord(s):
    s = str(s).strip('()').split(',')
    return float(s[0]), float(s[1])

print("Fitting KMeans hubs on real data...")
riders = pd.read_excel('riders.xlsx')
riders[['px','py']] = pd.DataFrame(
    riders['pickup_location'].apply(parse_coord).tolist(),
    index=riders.index)
X = riders[['px','py']].values

HUB_CONFIGS = {}
for k in range(1, 11):
    km    = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    km.fit(X)
    sizes = np.bincount(km.labels_)
    order = np.argsort(-sizes)
    HUB_CONFIGS[k] = km.cluster_centers_[order]
print("Done.\n")


Fitting KMeans hubs on real data...
Done.



In [28]:

# =====================================================
# LOCATION FUNCTIONS
# =====================================================

def driver_initial_location():
    return np.array([np.clip(np.random.normal(9.97,  4.36), 0, AREA_SIZE),
                     np.clip(np.random.normal(11.51, 4.34), 0, AREA_SIZE)])

def rider_pickup_location():
    return np.array([np.clip(skewnorm.rvs(a= 1.80, loc= 4.18, scale=5.97), 0, AREA_SIZE),
                     np.clip(skewnorm.rvs(a=-2.49, loc=17.05, scale=6.31), 0, AREA_SIZE)])

def rider_dropoff_location():
    return np.array([np.clip(skewnorm.rvs(a= -1.65, loc=15.51, scale=6.24), 0, AREA_SIZE),
                     np.clip(skewnorm.rvs(a=-14.26, loc=19.50, scale=7.50), 0, AREA_SIZE)])


In [29]:

# =====================================================
# DRIVER & RIDER
# =====================================================

class Driver:
    def __init__(self, did, loc, ou):
        self.id=did; self.location=loc; self.online_until=ou
        self.earnings=0; self.status='idle'; self.busy_time=0

class Rider:
    def __init__(self, rid, o, d, arr, ab):
        self.id=rid; self.origin=o; self.destination=d
        self.arrival_time=arr; self.abandon_time=ab; self.matched=False


In [ ]:

# =====================================================
# SIMULATION
# =====================================================

class Simulation:

    def __init__(self, hubs, lam=0.0):
        self.hubs = hubs
        self.lam  = lam

        self.current_time     = 0
        self.event_list       = []
        self.idle_drivers     = {}
        self.busy_drivers     = {}
        self.waiting_riders   = {}
        self.driver_counter   = 0
        self.rider_counter    = 0
        self.total_riders     = 0
        self.abandoned_riders = 0
        self.total_queue_wait = 0
        self.total_true_wait  = 0
        self.driver_log       = []

    def exp_time(self, r): return np.random.exponential(1 / r)
    def dist(self, a, b):  return np.linalg.norm(np.array(a) - np.array(b))
    def ttime(self, d):
        mu = d / SPEED; return np.random.uniform(0.8 * mu, 1.2 * mu)
    def schedule(self, t, e, d=None):
        heapq.heappush(self.event_list, (t, e, d))

    def nearest_hub_dist(self, loc):
        return min(np.linalg.norm(loc - h) for h in self.hubs)

    # ── Matching: GH combined score = scaled_1 - scaled_2 ──────────
    # score1 = dist(driver→pickup) / remaining_patience   [urgency]
    # score2 = dist(driver→pickup) + λ × dist(dropoff→hub) [positioning]
    # min-max scale both across all waiting riders, then subtract
    # pick best (driver, rider) pair globally across all idle drivers

    def _raw_scores(self, driver):
        """Compute raw score1 and score2 for every waiting rider
        from the perspective of a given driver."""
        raw = {}
        for rid, rider in self.waiting_riders.items():
            d2p               = self.dist(driver.location, rider.origin)
            remaining_patience = max(rider.abandon_time - self.current_time, 1e-6)
            s1 = d2p / remaining_patience                               # urgency
            s2 = d2p + self.lam * self.nearest_hub_dist(rider.destination)  # hub
            raw[rid] = (s1, s2)
        return raw

    def _best_rider(self, driver):
        """Min-max scale both scores across all riders and return
        the rider id with the lowest Score_new = scaled_1 - scaled_2."""
        raw = self._raw_scores(driver)

        all_s1 = [v[0] for v in raw.values()]
        all_s2 = [v[1] for v in raw.values()]
        min_s1, max_s1 = min(all_s1), max(all_s1)
        min_s2, max_s2 = min(all_s2), max(all_s2)

        eps = 1e-9  # prevents division by zero without distorting scale

        def new_score(rid):
            s1, s2   = raw[rid]
            scaled_1 = (s1 - min_s1) / (max_s1 - min_s1 + eps)
            scaled_2 = (s2 - min_s2) / (max_s2 - min_s2 + eps)
            return scaled_1 - scaled_2  # GH's idea

        return min(raw, key=new_score)

    def try_match(self):
        if not self.idle_drivers or not self.waiting_riders: return

        # ── Find best (driver, rider) pair across ALL idle drivers ──
        # For each idle driver compute their best rider and score,
        # then pick the globally best pair — avoids arbitrary first-driver bias
        best_score = float('inf')
        best_did   = None
        best_rid   = None

        for did, driver in self.idle_drivers.items():
            raw = self._raw_scores(driver)

            all_s1 = [v[0] for v in raw.values()]
            all_s2 = [v[1] for v in raw.values()]
            min_s1, max_s1 = min(all_s1), max(all_s1)
            min_s2, max_s2 = min(all_s2), max(all_s2)
            eps = 1e-9

            for rid in raw:
                s1, s2   = raw[rid]
                scaled_1 = (s1 - min_s1) / (max_s1 - min_s1 + eps)
                scaled_2 = (s2 - min_s2) / (max_s2 - min_s2 + eps)
                score    = scaled_1 - scaled_2

                if score < best_score:
                    best_score = score
                    best_did   = did
                    best_rid   = rid

        driver = self.idle_drivers[best_did]
        rider  = self.waiting_riders[best_rid]
        rider.matched = True

        pd_ = self.dist(driver.location, rider.origin)
        td  = self.dist(rider.origin, rider.destination)
        pt  = self.ttime(pd_); tt2 = self.ttime(td); tt = pt + tt2

        qw = self.current_time - rider.arrival_time
        self.total_queue_wait += qw
        self.total_true_wait  += qw + pt

        earn = (BASE_FARE + FARE_PER_MILE * td) - \
               PETROL_COST_PER_MILE * (pd_ + td)

        driver.status = 'busy'; driver.busy_time += tt
        self.busy_drivers[best_did] = driver
        del self.idle_drivers[best_did]; del self.waiting_riders[best_rid]
        self.schedule(self.current_time + tt, 'DROPOFF',
                      (best_did, rider.destination, earn))

    def handle_driver_arrival(self):
        self.driver_counter += 1; did = self.driver_counter
        loc = driver_initial_location()
        ou  = self.current_time + np.random.uniform(6, 8)
        self.idle_drivers[did] = Driver(did, loc, ou)
        self.driver_log.append({'driver_id': did, 'arrival_time': self.current_time,
                                'online_until': ou, 'earnings': 0, 'busy_time': 0})
        self.schedule(ou, 'DRIVER_OFFLINE', did)
        self.schedule(self.current_time + self.exp_time(DRIVER_RATE), 'DRIVER_ARRIVAL')
        self.try_match()

    def handle_rider_arrival(self):
        self.rider_counter += 1; rid = self.rider_counter
        self.total_riders += 1
        o  = rider_pickup_location(); d = rider_dropoff_location()
        ab = self.current_time + self.exp_time(PATIENCE_RATE)
        self.waiting_riders[rid] = Rider(rid, o, d, self.current_time, ab)
        self.schedule(ab, 'RIDER_ABANDON', rid)
        self.schedule(self.current_time + self.exp_time(RIDER_RATE), 'RIDER_ARRIVAL')
        self.try_match()

    def handle_dropoff(self, data):
        did, nloc, earn = data; driver = self.busy_drivers[did]
        driver.earnings += earn; driver.location = nloc; driver.status = 'idle'
        for d in self.driver_log:
            if d['driver_id'] == did:
                d['earnings'] += earn; d['busy_time'] = driver.busy_time; break
        del self.busy_drivers[did]
        if self.current_time < driver.online_until:
            self.idle_drivers[did] = driver
        self.try_match()

    def run(self):
        self.schedule(0, 'DRIVER_ARRIVAL'); self.schedule(0, 'RIDER_ARRIVAL')
        while self.event_list and self.current_time < SIM_TIME:
            self.current_time, et, data = heapq.heappop(self.event_list)
            if   et == 'DRIVER_ARRIVAL': self.handle_driver_arrival()
            elif et == 'RIDER_ARRIVAL':  self.handle_rider_arrival()
            elif et == 'DROPOFF':        self.handle_dropoff(data)
            elif et == 'RIDER_ABANDON':
                if data in self.waiting_riders:
                    self.abandoned_riders += 1; del self.waiting_riders[data]
            elif et == 'DRIVER_OFFLINE':
                self.idle_drivers.pop(data, None)
        return self.collect_results()

    def collect_results(self):
        matched = self.total_riders - self.abandoned_riders
        df      = pd.DataFrame(self.driver_log)
        df['online_duration']  = df['online_until'] - df['arrival_time']
        df['busy_time_capped'] = df[['busy_time','online_duration']].min(axis=1)
        df['idle_time']        = df['online_duration'] - df['busy_time_capped']

        p  = df['earnings'].values; N = len(p); mp = p.mean()
        vp = np.sum((p - mp)**2) / (N - 1) if N > 1 else 0
        ps = np.sort(p)
        cvar     = np.mean(ps[:max(1, int(np.floor(CVAR_ALPHA * N)))])
        cvar_med = np.mean(ps[:max(1, int(np.floor(0.5 * N)))])

        return {
            'abandonment_rate'  : self.abandoned_riders / self.total_riders,
            'avg_queue_wait'    : self.total_queue_wait / matched if matched > 0 else 0,
            'avg_true_wait'     : self.total_true_wait  / matched if matched > 0 else 0,
            'matched_riders'    : matched,
            'total_riders'      : self.total_riders,
            'abandoned_riders'  : self.abandoned_riders,
            'avg_earnings'      : mp,
            'total_earnings'    : df['earnings'].sum(),
            'avg_profit_per_hr' : df['earnings'].sum() / df['online_duration'].sum(),
            'avg_idle_time'     : df['idle_time'].mean(),
            'variance'          : vp,
            'fairness_cv'       : np.sqrt(vp) / mp if mp > 0 else 0,
            'cvar'              : cvar,
            'delta_cvar_internal': cvar - cvar_med,
            '_driver_df'        : df,
        }

# =====================================================
# BASELINE (no hubs — pure closest distance)
# =====================================================

class BaselineSim(Simulation):
    """Baseline: pure closest-distance matching.
    Overrides try_match to avoid the GH scoring logic — picks
    the best (driver, rider) pair globally by minimum pickup distance."""

    def try_match(self):
        if not self.idle_drivers or not self.waiting_riders: return

        # Find globally best (driver, rider) pair by shortest pickup distance
        best_score = float('inf')
        best_did   = None
        best_rid   = None

        for did, driver in self.idle_drivers.items():
            for rid, rider in self.waiting_riders.items():
                d = self.dist(driver.location, rider.origin)
                if d < best_score:
                    best_score = d
                    best_did   = did
                    best_rid   = rid

        driver = self.idle_drivers[best_did]
        rider  = self.waiting_riders[best_rid]
        rider.matched = True

        pd_ = self.dist(driver.location, rider.origin)
        td  = self.dist(rider.origin, rider.destination)
        pt  = self.ttime(pd_); tt2 = self.ttime(td); tt = pt + tt2

        qw = self.current_time - rider.arrival_time
        self.total_queue_wait += qw
        self.total_true_wait  += qw + pt

        earn = (BASE_FARE + FARE_PER_MILE * td) - \
               PETROL_COST_PER_MILE * (pd_ + td)

        driver.status = 'busy'; driver.busy_time += tt
        self.busy_drivers[best_did] = driver
        del self.idle_drivers[best_did]; del self.waiting_riders[best_rid]
        self.schedule(self.current_time + tt, 'DROPOFF',
                      (best_did, rider.destination, earn))

print("Running Baseline...")
np.random.seed(42)
res_base = BaselineSim(hubs=np.array([[0,0]]), lam=0).run()
res_base['_driver_df'].to_csv(f"{OUTPUT_DIR}/data_baseline.csv", index=False)
print(f"  Abandon: {res_base['abandonment_rate']*100:.2f}%  "
      f"Earn: £{res_base['avg_earnings']:.2f}\n")

bv = res_base['variance']
bc = res_base['cvar']

# =====================================================
# MAIN LOOP — λ × K
# =====================================================

summary_rows = [{
    'lambda'            : 0.0,
    'n_hubs'            : 0,
    'config'            : 'baseline',
    'abandonment_pct'   : res_base['abandonment_rate'] * 100,
    'avg_queue_wait_min': res_base['avg_queue_wait'] * 60,
    'avg_true_wait_min' : res_base['avg_true_wait']  * 60,
    'matched_riders'    : res_base['matched_riders'],
    'avg_earnings'      : res_base['avg_earnings'],
    'total_earnings'    : res_base['total_earnings'],
    'avg_profit_per_hr' : res_base['avg_profit_per_hr'],
    'avg_idle_time'     : res_base['avg_idle_time'],
    'variance'          : res_base['variance'],
    'fairness_cv'       : res_base['fairness_cv'],
    'cvar'              : res_base['cvar'],
    'fairness_effect'   : 0.0,
    'delta_cvar_cross'  : 0.0,
    'delta_profit'      : 0.0,
}]

total_runs = len(LAM_VALUES) * len(list(K_VALUES))
run_num    = 0

for lam in LAM_VALUES:
    print(f"{'='*65}")
    print(f"  λ = {lam}")
    print(f"{'='*65}")
    print(f"  {'Config':<14} {'Abandon%':>9} {'Queue(min)':>11} "
          f"{'True(min)':>10} {'AvgEarn£':>9} {'TotalEarn£':>12} {'FairnessEff':>12} {'ΔCVaR£':>9}")
    print(f"  {'-'*90}")

    lam_rows = []   # collect all K rows for this lambda

    for k in K_VALUES:
        run_num += 1
        config = f"lam{lam:.0f}_kmean_{k}"

        np.random.seed(42)
        sim = Simulation(hubs=HUB_CONFIGS[k], lam=lam)
        res = sim.run()

        # ── Cross-run fairness vs baseline ─────────────────
        fe           = 1 - res['variance'] / bv if bv > 0 else 0
        delta_cvar_x = res['cvar'] - bc
        delta_profit = res['avg_earnings'] - res_base['avg_earnings']

        row = {
            'lambda'            : lam,
            'n_hubs'            : k,
            'config'            : config,
            'abandonment_pct'   : res['abandonment_rate'] * 100,
            'avg_queue_wait_min': res['avg_queue_wait'] * 60,
            'avg_true_wait_min' : res['avg_true_wait']  * 60,
            'matched_riders'    : res['matched_riders'],
            'avg_earnings'      : res['avg_earnings'],
            'total_earnings'    : res['total_earnings'],
            'avg_profit_per_hr' : res['avg_profit_per_hr'],
            'avg_idle_time'     : res['avg_idle_time'],
            'variance'          : res['variance'],
            'fairness_cv'       : res['fairness_cv'],
            'cvar'              : res['cvar'],
            'fairness_effect'   : fe,
            'delta_cvar_cross'  : delta_cvar_x,
            'delta_profit'      : delta_profit,
        }
        summary_rows.append(row)
        lam_rows.append(row)

        print(f"  {config:<14} "
              f"{res['abandonment_rate']*100:>8.2f}% "
              f"{res['avg_queue_wait']*60:>11.3f} "
              f"{res['avg_true_wait']*60:>10.3f} "
              f"{res['avg_earnings']:>9.2f} "
              f"{res['total_earnings']:>12.2f} "
              f"{fe:>+12.4f} "
              f"{delta_cvar_x:>+9.2f}   "
              f"[{run_num}/{total_runs}]")

    # ── Save one CSV per lambda (all K values together) ────
    lam_df = pd.DataFrame(lam_rows)
    lam_df.to_csv(f"{OUTPUT_DIR}/data_lam{lam:.0f}.csv", index=False)
    print(f"  -> Saved: data_lam{lam:.0f}.csv
")

# =====================================================
# SAVE SUMMARY
# =====================================================

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(f"{OUTPUT_DIR}/summary_lambda_kmeans.csv", index=False)

# =====================================================
# PRINT BEST PER LAMBDA
# =====================================================

print("\n" + "="*75)
print("  BEST K PER LAMBDA (lowest abandonment)")
print("="*75)
print(f"  {'λ':>4} {'Best K':>7} {'Config':<18} "
      f"{'Abandon%':>9} {'AvgEarn£':>9} {'FairnessEff':>12} {'ΔCVaR£':>9}")
print("-"*75)
non_base = summary_df[summary_df['config'] != 'baseline']
for lam in LAM_VALUES:
    subset = non_base[non_base['lambda'] == lam]
    best   = subset.loc[subset['abandonment_pct'].idxmin()]
    print(f"  {lam:>4.1f} {int(best['n_hubs']):>7} {best['config']:<18} "
          f"{best['abandonment_pct']:>8.2f}% "
          f"{best['avg_earnings']:>9.2f} "
          f"{best['fairness_effect']:>+12.4f} "
          f"{best['delta_cvar_cross']:>+9.2f}")
print("="*75)

# Overall best
best_overall = non_base.loc[non_base['abandonment_pct'].idxmin()]
print(f"\n  Overall best abandonment: {best_overall['config']}  "
      f"→  {best_overall['abandonment_pct']:.2f}%")

print(f"\nAll files saved to: {OUTPUT_DIR}/")
print(f"  data_baseline.csv")
print(f"  data_lam[1-5]_kmean_[1-10].csv  (50 files)")
print(f"  summary_lambda_kmeans.csv")

In [31]:

# =====================================================
# BASELINE (no hubs — pure closest distance)
# =====================================================

class BaselineSim(Simulation):
    """Baseline: pure closest-distance matching.
    Overrides try_match to avoid the GH scoring logic — picks
    the best (driver, rider) pair globally by minimum pickup distance."""

    def try_match(self):
        if not self.idle_drivers or not self.waiting_riders: return

        # Find globally best (driver, rider) pair by shortest pickup distance
        best_score = float('inf')
        best_did   = None
        best_rid   = None

        for did, driver in self.idle_drivers.items():
            for rid, rider in self.waiting_riders.items():
                d = self.dist(driver.location, rider.origin)
                if d < best_score:
                    best_score = d
                    best_did   = did
                    best_rid   = rid

        driver = self.idle_drivers[best_did]
        rider  = self.waiting_riders[best_rid]
        rider.matched = True

        pd_ = self.dist(driver.location, rider.origin)
        td  = self.dist(rider.origin, rider.destination)
        pt  = self.ttime(pd_); tt2 = self.ttime(td); tt = pt + tt2

        qw = self.current_time - rider.arrival_time
        self.total_queue_wait += qw
        self.total_true_wait  += qw + pt

        earn = (BASE_FARE + FARE_PER_MILE * td) - \
               PETROL_COST_PER_MILE * (pd_ + td)

        driver.status = 'busy'; driver.busy_time += tt
        self.busy_drivers[best_did] = driver
        del self.idle_drivers[best_did]; del self.waiting_riders[best_rid]
        self.schedule(self.current_time + tt, 'DROPOFF',
                      (best_did, rider.destination, earn))

print("Running Baseline...")
np.random.seed(42)
res_base = BaselineSim(hubs=np.array([[0,0]]), lam=0).run()
res_base['_driver_df'].to_csv(f"{OUTPUT_DIR}/data_baseline.csv", index=False)
print(f"  Abandon: {res_base['abandonment_rate']*100:.2f}%  "
      f"Earn: £{res_base['avg_earnings']:.2f}\n")

bv = res_base['variance']
bc = res_base['cvar']


Running Baseline...
  Abandon: 2.16%  Earn: £118.39



In [32]:

# =====================================================
# MAIN LOOP — λ × K
# =====================================================

summary_rows = [{
    'lambda'            : 0.0,
    'n_hubs'            : 0,
    'config'            : 'baseline',
    'abandonment_pct'   : res_base['abandonment_rate'] * 100,
    'avg_queue_wait_min': res_base['avg_queue_wait'] * 60,
    'avg_true_wait_min' : res_base['avg_true_wait']  * 60,
    'matched_riders'    : res_base['matched_riders'],
    'avg_earnings'      : res_base['avg_earnings'],
    'total_earnings'    : res_base['total_earnings'],
    'avg_profit_per_hr' : res_base['avg_profit_per_hr'],
    'avg_idle_time'     : res_base['avg_idle_time'],
    'variance'          : res_base['variance'],
    'fairness_cv'       : res_base['fairness_cv'],
    'cvar'              : res_base['cvar'],
    'fairness_effect'   : 0.0,
    'delta_cvar_cross'  : 0.0,
    'delta_profit'      : 0.0,
}]

total_runs = len(LAM_VALUES) * len(list(K_VALUES))
run_num    = 0

for lam in LAM_VALUES:
    print(f"{'='*65}")
    print(f"  λ = {lam}")
    print(f"{'='*65}")
    print(f"  {'Config':<14} {'Abandon%':>9} {'Queue(min)':>11} "
          f"{'True(min)':>10} {'AvgEarn£':>9} {'TotalEarn£':>12} {'FairnessEff':>12} {'ΔCVaR£':>9}")
    print(f"  {'-'*90}")

    lam_rows = []   # collect all K rows for this lambda

    for k in K_VALUES:
        run_num += 1
        config = f"lam{lam:.0f}_kmean_{k}"

        np.random.seed(42)
        sim = Simulation(hubs=HUB_CONFIGS[k], lam=lam)
        res = sim.run()

        # ── Cross-run fairness vs baseline ─────────────────
        fe           = 1 - res['variance'] / bv if bv > 0 else 0
        delta_cvar_x = res['cvar'] - bc
        delta_profit = res['avg_earnings'] - res_base['avg_earnings']

        row = {
            'lambda'            : lam,
            'n_hubs'            : k,
            'config'            : config,
            'abandonment_pct'   : res['abandonment_rate'] * 100,
            'avg_queue_wait_min': res['avg_queue_wait'] * 60,
            'avg_true_wait_min' : res['avg_true_wait']  * 60,
            'matched_riders'    : res['matched_riders'],
            'avg_earnings'      : res['avg_earnings'],
            'total_earnings'    : res['total_earnings'],
            'avg_profit_per_hr' : res['avg_profit_per_hr'],
            'avg_idle_time'     : res['avg_idle_time'],
            'variance'          : res['variance'],
            'fairness_cv'       : res['fairness_cv'],
            'cvar'              : res['cvar'],
            'fairness_effect'   : fe,
            'delta_cvar_cross'  : delta_cvar_x,
            'delta_profit'      : delta_profit,
        }
        summary_rows.append(row)
        lam_rows.append(row)

        print(f"  {config:<14} "
              f"{res['abandonment_rate']*100:>8.2f}% "
              f"{res['avg_queue_wait']*60:>11.3f} "
              f"{res['avg_true_wait']*60:>10.3f} "
              f"{res['avg_earnings']:>9.2f} "
              f"{res['total_earnings']:>12.2f} "
              f"{fe:>+12.4f} "
              f"{delta_cvar_x:>+9.2f}   "
              f"[{run_num}/{total_runs}]")

    # ── Save one CSV per lambda (all K values together) ────
    lam_str = f"{lam:.1f}".replace(".", "_")

    lam_df = pd.DataFrame(lam_rows)
    lam_df.to_csv(f"{OUTPUT_DIR}/data_lam{lam_str}.csv", index=False)

    print(f"Saved: data_lam{lam_str}.csv")

  λ = 1.0
  Config          Abandon%  Queue(min)  True(min)  AvgEarn£   TotalEarn£  FairnessEff    ΔCVaR£
  ------------------------------------------------------------------------------------------
  lam1_kmean_3       1.87%       0.152     14.960    117.69    612484.75      +0.0538     +3.70   [1/40]
Saved: data_lam1_0.csv
  λ = 1.1
  Config          Abandon%  Queue(min)  True(min)  AvgEarn£   TotalEarn£  FairnessEff    ΔCVaR£
  ------------------------------------------------------------------------------------------
  lam1_kmean_3       1.96%       0.156     14.995    118.51    613773.50      -0.0002     +1.28   [2/40]
Saved: data_lam1_1.csv
  λ = 1.2
  Config          Abandon%  Queue(min)  True(min)  AvgEarn£   TotalEarn£  FairnessEff    ΔCVaR£
  ------------------------------------------------------------------------------------------
  lam1_kmean_3       1.80%       0.159     15.219    120.15    614574.01      +0.0851     +7.63   [3/40]
Saved: data_lam1_2.csv
  λ = 1.3
  Config 

In [33]:

# =====================================================
# SAVE SUMMARY
# =====================================================

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(f"{OUTPUT_DIR}/summary_lambda_kmeans.csv", index=False)
